# EDA & Statistics — Northwind Dataset

**Goal:** Full EDA pipeline on a real business dataset.

**Tools:** Pandas · Seaborn · Matplotlib · SciPy

**Author:** Santiago Acosta

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

In [ ]:
df = pd.read_csv('../data/northwind_clean.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Data Inspection

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Duplicates ---')
print(f'{df.duplicated().sum()} duplicate rows')

df['revenue'] = df['unit_price'] * df['quantity'] * (1 - df['discount'])
df.describe().round(2)

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df['revenue'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Revenue Distribution')

sns.histplot(df['quantity'], kde=True, ax=axes[1], color='teal')
axes[1].set_title('Quantity Distribution')

sns.histplot(df['discount'], kde=True, ax=axes[2], color='slategray')
axes[2].set_title('Discount Distribution')

plt.tight_layout()
plt.savefig('../assets/univariate_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

for col in ['revenue', 'quantity', 'discount']:
    print(f'{col:12s} | skewness: {df[col].skew():6.2f} | kurtosis: {df[col].kurtosis():6.2f}')

In [ ]:
cat_revenue = df.groupby('category_name')['revenue'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(x=cat_revenue.values, y=cat_revenue.index, palette='Blues_r')
plt.title('Total Revenue by Category')
plt.xlabel('Revenue (USD)')
plt.tight_layout()
plt.savefig('../assets/revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(data=df, x='discount', y='revenue', ax=axes[0],
            scatter_kws={'alpha': 0.3, 'color': 'steelblue'},
            line_kws={'color': 'red'})
axes[0].set_title('Discount vs Revenue')

sns.regplot(data=df, x='quantity', y='revenue', ax=axes[1],
            scatter_kws={'alpha': 0.3, 'color': 'teal'},
            line_kws={'color': 'red'})
axes[1].set_title('Quantity vs Revenue')

plt.tight_layout()
plt.savefig('../assets/bivariate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
monthly = df.groupby(df['order_date'].dt.to_period('M'))['revenue'].sum().reset_index()
monthly['order_date'] = monthly['order_date'].astype(str)

plt.figure(figsize=(12, 4))
sns.lineplot(data=monthly, x='order_date', y='revenue', color='steelblue', linewidth=2)
plt.xticks(rotation=45)
plt.title('Monthly Revenue Trend')
plt.tight_layout()
plt.savefig('../assets/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
numeric_cols = ['revenue', 'quantity', 'unit_price', 'discount']
corr_matrix = df[numeric_cols].corr(method='pearson')

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Pearson Correlation Matrix')
plt.tight_layout()
plt.savefig('../assets/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

r, p = stats.pearsonr(df['discount'], df['revenue'])
print(f'Discount vs Revenue — Pearson r: {r:.3f}, p-value: {p:.4f}')

## 6. Outlier Detection

In [ ]:
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = df[(df['revenue'] < lower) | (df['revenue'] > upper)]
print(f'IQR bounds: [{lower:.2f}, {upper:.2f}]')
print(f'Outliers: {len(outliers_iqr)} ({len(outliers_iqr)/len(df)*100:.1f}% of records)')

df['z_unit_price'] = np.abs(stats.zscore(df['unit_price']))
outliers_z = df[df['z_unit_price'] > 3]
print(f'Z-score outliers (unit_price > 3σ): {len(outliers_z)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(y=df['revenue'], ax=axes[0], color='steelblue')
axes[0].set_title('Revenue — Boxplot')
sns.boxplot(y=df['unit_price'], ax=axes[1], color='teal')
axes[1].set_title('Unit Price — Boxplot')
plt.tight_layout()
plt.savefig('../assets/outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Statistical Summaries

In [ ]:
summary = df.groupby('category_name')['revenue'].agg(
    total='sum', mean='mean', median='median', std='std', count='count'
).sort_values('total', ascending=False).round(2)
summary['% of total'] = (summary['total'] / summary['total'].sum() * 100).round(1)
print(summary)

In [ ]:
top_products = df.groupby('product_name')['revenue'].sum()\
    .sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=top_products, x='revenue', y='product_name', palette='Blues_r')
plt.title('Top 10 Products by Revenue')
plt.tight_layout()
plt.savefig('../assets/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

customer_rev = df.groupby('customer_id')['revenue'].sum().sort_values(ascending=False)
cumulative = customer_rev.cumsum() / customer_rev.sum() * 100
top_20pct = int(len(customer_rev) * 0.2)
print(f'Top 20% of customers ({top_20pct}) account for {cumulative.iloc[top_20pct-1]:.1f}% of revenue')

## 8. Key Findings

| Finding | Detail |
|---|---|
| Revenue is right-skewed | Most orders are small; a few bulk orders drive disproportionate revenue |
| Discount effect | Weak positive correlation with quantity, negative impact on per-order revenue |
| Category concentration | Top 3 categories account for ~60% of total revenue |
| Outliers are valid | High-revenue outliers correspond to bulk B2B purchases, not data errors |
| Customer concentration | Top 20% of customers drive majority of revenue (Pareto principle holds) |